# Heart Disease Prediction using Machine Learning

End-to-end notebook: EDA → Feature Engineering → Preprocessing (demo) → Modeling → Evaluation.

> **Design Note**: EDA is performed on **raw** data (`heart.csv`). Preprocessing steps are explained narratively. Modeling uses `heart_processed.csv` (output of `src/preprocess.py`) to ensure **results are identical** to the production script.

| | |
|---|---|
| **Dataset** | 918 observations, 11 clinical features (Kaggle – fedesoriano) |
| **Target** | `HeartDisease` (0 = No, 1 = Yes) |
| **Model** | Random Forest + GridSearchCV |

## 1️⃣ Setup & Import Libraries

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gmean
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, cohen_kappa_score,
    classification_report
)
import joblib
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ Libraries imported')

## 2️⃣ Exploratory Data Analysis

EDA is performed on **raw data** (`heart.csv`) before any preprocessing.

### 2.1 Load Raw Dataset

In [ ]:
df = pd.read_csv('../Data/heart.csv')
print('Shape:', df.shape)
display(df.head())

### 2.2 Data Structure

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nDescriptive Statistics:')
display(df.describe())

### 2.3 Data Quality Check

In [ ]:
print(f'Duplicate rows : {df.duplicated().sum()}')
print(f'Missing values :\n{df.isna().sum()}')
num_cols = df.select_dtypes(include='number').columns
zero_counts = (df[num_cols] == 0).sum()
print(f'\nZero values in numeric cols:')
print(zero_counts[zero_counts > 0])

### 2.4 Target Distribution

In [ ]:
target_counts = df['HeartDisease'].value_counts()
print(target_counts)
print(df['HeartDisease'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'], edgecolor='black')
axes[0].set_title('Heart Disease Count', fontweight='bold')
axes[0].set_xticklabels(['No Disease','Has Disease'], rotation=0)
axes[0].set_ylabel('Count')
target_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                   colors=['#2ecc71','#e74c3c'], labels=['No Disease','Has Disease'])
axes[1].set_title('Heart Disease %', fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout(); plt.show()

### 2.5 Numeric Feature Distributions

In [ ]:
num_features = ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_features):
    for label, color in zip([0,1], ['#2ecc71','#e74c3c']):
        axes[i].hist(df[df['HeartDisease']==label][col], bins=25, alpha=0.6,
                     color=color, label=f'HD={label}')
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend()
axes[-1].set_visible(False)
plt.suptitle('Numeric Feature Distributions by Target', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 2.6 Categorical Features vs Target

In [ ]:
cat_features = df.select_dtypes(include='object').columns.tolist()
print('Categorical features:', cat_features)
n = len(cat_features)
fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
if n == 1: axes = [axes]
for ax, col in zip(axes, cat_features):
    ct = pd.crosstab(df[col], df['HeartDisease'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c'], edgecolor='black')
    ax.set_title(f'{col} vs HeartDisease', fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.legend(['No Disease','Has Disease'])
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

### 2.7 Correlation Matrix (Raw Features)

In [ ]:
corr_cols = ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak','HeartDisease']
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix – Raw Features', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('\nCorrelation with HeartDisease:')
print(corr['HeartDisease'].sort_values(ascending=False))

## 3️⃣ Feature Engineering

### Why HR_Ratio?

A 30-year-old with MaxHR=160 is very different from a 70-year-old with the same value. **HR_Ratio** normalizes MaxHR relative to age:

$$HR\_Ratio = \\frac{MaxHR}{220 - Age}$$

A value near 1.0 means the heart can reach its age-predicted maximum capacity.

In [ ]:
# Demo: effect of HR_Ratio on correlation
df_demo = df.copy()
df_demo['HR_Ratio'] = df_demo['MaxHR'] / (220 - df_demo['Age'])

print('Correlation  MaxHR vs HeartDisease:', df_demo['MaxHR'].corr(df_demo['HeartDisease']).round(4))
print('Correlation HR_Ratio vs HeartDisease:', df_demo['HR_Ratio'].corr(df_demo['HeartDisease']).round(4))
print('\nSample HR_Ratio values:')
display(df_demo[['Age','MaxHR','HR_Ratio','HeartDisease']].head(8))

# Correlation after feature engineering
corr_cols2 = ['Age','RestingBP','Cholesterol','Oldpeak','HR_Ratio','HeartDisease']
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_demo[corr_cols2].corr(), annot=True, cmap='coolwarm', fmt='.2f',
            ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix – After HR_Ratio Engineering', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4️⃣ Preprocessing Overview

> The preprocessing pipeline is **executed** by `src/preprocess.py` and produces `Data/heart_processed.csv`. This section explains each step.

| Step | Detail |
|---|---|
| **Imputation** | `Cholesterol==0` & `RestingBP==0` → replaced with non-zero median |
| **Label Encoding** | Binary features (2 categories): `Sex`, `ExerciseAngina` |
| **One-Hot Encoding** | Multiclass features (>2): `ChestPainType`, `RestingECG`, `ST_Slope` |
| **Standard Scaling** | `Age`, `RestingBP`, `Cholesterol`, `Oldpeak`, `HR_Ratio` |

In [ ]:
# Verify preprocessing output
df_proc = pd.read_csv('../Data/heart_processed.csv')
print('Processed data shape:', df_proc.shape)
print('\nColumns:')
print(df_proc.columns.tolist())
print('\nSample:')
display(df_proc.head())

## 5️⃣ Modeling – Random Forest + GridSearchCV

Modeling uses **`heart_processed.csv`** directly to ensure results are identical to `src/modeling.py`.

### 5.1 Load Processed Data & Split

In [ ]:
X = df_proc.drop('HeartDisease', axis=1)
y = df_proc['HeartDisease']
print(f'Features : {X.shape}')
print(f'Target   : {y.shape}')
print(f'\nClass distribution:\n{y.value_counts()}')

# Split: 80% CV, 20% blind test (identical to modeling.py)
X_cv, X_test, y_cv, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nCV   : {X_cv.shape}')
print(f'Test : {X_test.shape}')

### 5.2 GridSearchCV Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators'    : [90, 100, 200, 300],
    'max_depth'       : [8, 10, 15],
    'min_samples_split': [25, 30]
}
print('Total combinations:', np.prod([len(v) for v in param_grid.values()]))

rf_base    = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(rf_base, param_grid, cv=5,
                            scoring='roc_auc', n_jobs=-1, verbose=1)
print('\n⏳ Training...')
grid_search.fit(X_cv, y_cv)

best_model  = grid_search.best_estimator_
best_params = grid_search.best_params_
print('\n✅ Best Parameters:', best_params)
print(f'Best CV ROC-AUC  : {grid_search.best_score_:.4f}')

## 6️⃣ Model Evaluation

### 6.1 Performance Metrics

In [ ]:
y_pred_cv    = best_model.predict(X_cv)
y_proba_cv   = best_model.predict_proba(X_cv)[:,1]
y_pred_test  = best_model.predict(X_test)
y_proba_test = best_model.predict_proba(X_test)[:,1]

def get_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tpr = tp/(tp+fn) if (tp+fn)>0 else 0
    tnr = tn/(tn+fp) if (tn+fp)>0 else 0
    return {
        'Accuracy' : accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall'   : recall_score(y_true, y_pred),
        'F1-Score' : f1_score(y_true, y_pred),
        'ROC-AUC'  : roc_auc_score(y_true, y_proba),
        'Kappa'    : cohen_kappa_score(y_true, y_pred),
        'G-Mean'   : gmean([tpr, tnr]),
        'CM'       : confusion_matrix(y_true, y_pred)
    }

m_cv   = get_metrics(y_cv,   y_pred_cv,   y_proba_cv)
m_test = get_metrics(y_test, y_pred_test, y_proba_test)

keys = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC','Kappa','G-Mean']
summary = pd.DataFrame({
    'CV (Train)' : {k: round(m_cv[k],4)   for k in keys},
    'Test (Blind)': {k: round(m_test[k],4) for k in keys}
}).T
print('='*60)
print('PERFORMANCE METRICS SUMMARY')
print('='*60)
display(summary)

### 6.2 Classification Report

In [ ]:
print('CV (Train):')
print(classification_report(y_cv, y_pred_cv, target_names=['No Disease','Has Disease']))
print('Test (Blind):')
print(classification_report(y_test, y_pred_test, target_names=['No Disease','Has Disease']))

### 6.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrices – RandomForest', fontsize=14, fontweight='bold')
for ax, cm, title, cmap in zip(
    axes,
    [m_cv['CM'], m_test['CM']],
    ['CV Data (Train)', 'Test Data (Blind)'],
    ['Blues', 'Greens']
):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['No Disease','Has Disease'],
                yticklabels=['No Disease','Has Disease'])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = m_test['CM'].ravel()
print(f'Test Set – TN:{tn}  FP:{fp}  FN:{fn}  TP:{tp}')
print(f'Sensitivity (Recall) : {tp/(tp+fn):.2%}')
print(f'Specificity           : {tn/(tn+fp):.2%}')

### 6.4 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for fpr, tpr, label, color in [
    (*roc_curve(y_cv, y_proba_cv)[:2], f'CV (AUC={m_cv["ROC-AUC"]:.4f})', 'steelblue'),
    (*roc_curve(y_test, y_proba_test)[:2], f'Test (AUC={m_test["ROC-AUC"]:.4f})', 'seagreen')
]:
    ax.plot(fpr, tpr, label=label, linewidth=2.5, color=color)
ax.plot([0,1],[0,1],'k--', linewidth=1.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves – RandomForest', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### 6.5 Feature Importance

In [ ]:
importances = best_model.feature_importances_
feat_names  = X.columns
top_idx     = np.argsort(importances)[-15:]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(range(len(top_idx)), importances[top_idx],
        color='steelblue', alpha=0.85, edgecolor='black')
ax.set_yticks(range(len(top_idx)))
ax.set_yticklabels([feat_names[i] for i in top_idx], fontsize=10)
ax.set_xlabel('Importance Score')
ax.set_title('Top 15 Most Important Features', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

## 7️⃣ Key Insights & Conclusion

### 📊 Ringkasan Performa Model

| Metrik | CV (Train) | Test (Blind) |
|---|---|---|
| Accuracy | 0.8842 | **0.8641** |
| Precision | 0.8707 | **0.8598** |
| Recall | 0.9286 | **0.9020** |
| F1-Score | 0.8987 | **0.8804** |
| ROC-AUC | 0.9676 | **0.9220** |
| Kappa | 0.7639 | **0.7234** |
| G-Mean | 0.8775 | **0.8585** |

### 💡 Key Insights

1. **Recall 90.20%** — the model catches 9 out of 10 heart disease cases. Critical in a medical context.
2. **ROC-AUC 0.922** — excellent discrimination between positive and negative classes.
3. **Small CV vs Test gap** (~3% AUC) — minimal overfitting, model generalizes well to unseen data.
4. **HR_Ratio** appears in top feature importance — validates the feature engineering approach.
5. **ST_Slope & ChestPainType** are the strongest predictors, consistent with clinical literature.

### ⚠️ Limitations

- Small dataset (918 samples) combined from 5 different source datasets.
- Model **does not replace** professional medical diagnosis.

> **Medical Disclaimer**: This notebook is for educational purposes only. Predictions should not be used as a substitute for professional medical advice.